# Credit Default Prediction — Part 2: Feature Engineering

**Goal:** Transform raw borrower data into a clean, model-ready feature set. Every decision here is documented with a business rationale — not just a technical one.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/cs-training.csv', index_col=0)
target = 'SeriousDlqin2yrs'
print(f'Raw shape: {df.shape}')

## 1. Outlier Capping

`RevolvingUtilizationOfUnsecuredLines` > 1.0 represents data entry errors (utilization cannot exceed 100%). We cap at 1.0.
Extreme income values are capped at the 99th percentile to reduce leverage from outliers without dropping rows.

In [ ]:
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)

income_cap = df['MonthlyIncome'].quantile(0.99)
df['MonthlyIncome'] = df['MonthlyIncome'].clip(upper=income_cap)

debt_cap = df['DebtRatio'].quantile(0.99)
df['DebtRatio'] = df['DebtRatio'].clip(upper=debt_cap)

## 2. Missing Value Imputation

- **MonthlyIncome (~20% missing):** Median imputed by age group — income is highly age-dependent. Also create a binary flag `income_missing` to let the model learn whether missing income itself is a risk signal.
- **NumberOfDependents (~3% missing):** Median imputed overall.

In [ ]:
# Flag before imputing
df['income_missing'] = df['MonthlyIncome'].isnull().astype(int)

# Age-group median imputation for income
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100],
                          labels=['<25', '25-34', '35-44', '45-54', '55-64', '65+'])
age_income_median = df.groupby('age_group')['MonthlyIncome'].median()
df['MonthlyIncome'] = df.apply(
    lambda row: age_income_median[row['age_group']] if pd.isnull(row['MonthlyIncome']) else row['MonthlyIncome'],
    axis=1
)

# Simple median for dependents
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

print(f'Missing after imputation: {df.isnull().sum().sum()}')

## 3. Engineered Features

Raw features are useful; derived features that encode borrower behavior are often more predictive.

In [ ]:
# Total delinquency events — captures cumulative payment stress
df['total_delinquencies'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

# Weighted delinquency score — later-stage delinquency is more severe
df['delinquency_severity_score'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] * 1 +
    df['NumberOfTime60-89DaysPastDueNotWorse'] * 2 +
    df['NumberOfTimes90DaysLate'] * 3
)

# Debt burden per credit line — normalized leverage signal
df['debt_per_credit_line'] = df['DebtRatio'] / (df['NumberOfOpenCreditLinesAndLoans'] + 1)

# Monthly income-to-debt ratio — ability to service debt
df['income_to_debt_ratio'] = df['MonthlyIncome'] / (df['DebtRatio'] * df['MonthlyIncome'] + 1)

print('Engineered features added:')
print(['total_delinquencies', 'delinquency_severity_score', 'debt_per_credit_line', 'income_to_debt_ratio'])

## 4. Final Feature Set & Train/Test Split

In [ ]:
feature_cols = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate',
    'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfDependents',
    'income_missing',
    'total_delinquencies',
    'delinquency_severity_score',
    'debt_per_credit_line',
    'income_to_debt_ratio'
]

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}')
print(f'Default rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

# Save for modeling notebook
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
print('Train/test sets saved.')